# 03. Preprocessing

## Цель
Подготовить данные к обучению baseline-модели классификации бухгалтерских документов.

## Задачи
1. Загрузить и подготовить train/test выборки.
2. Выполнить минимальную очистку признаков.
3. Сформировать целевую переменную `Account_Number`.
4. Отобрать признаки для baseline-модели.
5. Построить preprocessing pipeline:
   - TF-IDF для текстовых признаков,
   - обработка числовых признаков.
6. Получить готовые матрицы признаков для обучения и тестирования.

In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
import joblib
from scipy import sparse

In [2]:
merged_df = pd.read_csv("../data/modified/merged_dataframe.csv")
merged_df

,Dataset,Date,Invoice,Supplier_Code,Supplier_Name,Invoice_Item,Item_Description,Item_Value,Item_Quantity,Total_Amount,Taxes_Credit_%,Taxes_Credit_Amount,Total_Net_Amount,Account_Number,Account_Description,Activity_Code,Activity_Description
0,Training,2019-08-15,91,10001,ABS & ABC Tax Consulting,1,Review of federal tax obligations,2000,1,2000,0,0.0,2000.0,4000010,Audit and Consulting,7020400,"Business management consulting, except special..."
1,Training,2019-08-15,186,10002,HJU Travel Agency,1,Assistance with flight and hotel arrangements,500,1,500,0,0.0,500.0,4000080,Other Third-Party Services,7911200,Travel agencies
2,Training,2019-08-15,275,10003,ABC Associated Lawyers,1,Legal advisory in labor lawsuit,2000,1,2000,0,0.0,2000.0,4000012,Legal Services,6911701,Legal services
3,Training,2019-08-15,135,10004,ATY Business Consulting,1,Consulting on zero-based budgeting,15000,1,15000,0,0.0,15000.0,4000010,Audit and Consulting,7020400,"Business management consulting, except special..."
4,Training,2019-08-15,221,10005,HBG Security and Surveillance,1,Provision of 2 security guards for gate,15000,1,15000,0,0.0,15000.0,4000015,Security and Surveillance,8020001,Electronic security system monitoring activiti...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
338,Test,2024-01-01,650,10023,MHG Industrial Tools,2,Tool ASR,4300,1,4300,20,860.0,3440.0,1001004,Industrial Tools,4612500,Commercial representatives and agents of fuels...
339,Test,2024-01-01,189,10024,PKM Quality Systems Auditing,1,ISO 9001 audit,25000,1,25000,0,0.0,25000.0,5000010,Audit and Consulting,4614100,Commercial representatives and agents of machi...
340,Test,2024-01-01,511,10025,PLM Industrial Hardware,1,Reinforced belt,260,2,520,20,104.0,416.0,5000034,Industrial Tools,6204000,IT consulting
341,Test,2024-01-01,685,10026,PRP Corporate Benefits,1,Meal voucher purchase,24000,1,24000,0,0.0,24000.0,5000003,Meal Voucher,7119704,Technical expertise services related to occupa...


In [3]:
def squeeze_text_column(x):
    return x.squeeze()

In [4]:
merged_df["Date"] = pd.to_datetime(merged_df["Date"], errors="coerce")
merged_df["Year"] = merged_df["Date"].dt.year
merged_df["Month"] = merged_df["Date"].dt.month
merged_df["Day"] = merged_df["Date"].dt.day
merged_df["Item_Description_Length"] = merged_df["Item_Description"].astype(str).str.len()
merged_df["Item_Description_Word_Count"] = merged_df["Item_Description"].astype(str).str.split().str.len()
merged_df["Has_Tax_Credit"] = (merged_df["Taxes_Credit_Amount"] > 0).astype(int)

In [5]:
text_cols_to_clean = ["Item_Description", "Activity_Description", "Supplier_Name"]
for col in text_cols_to_clean:
    merged_df[col] = (
        merged_df[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

In [6]:
train_df = merged_df[merged_df["Dataset"] == "Training"].copy()
test_df = merged_df[merged_df["Dataset"] == "Test"].copy()

In [7]:
text_features = [
    "Item_Description",
    "Activity_Description"
]
numeric_features = [
    "Item_Value",
    "Item_Quantity",
    "Taxes_Credit_%",
    "Taxes_Credit_Amount",
    "Month",
    "Item_Description_Length",
    "Item_Description_Word_Count",
    "Has_Tax_Credit"
]
selected_features = text_features + numeric_features
selected_features

['Item_Description',
 'Activity_Description',
 'Item_Value',
 'Item_Quantity',
 'Taxes_Credit_%',
 'Taxes_Credit_Amount',
 'Month',
 'Item_Description_Length',
 'Item_Description_Word_Count',
 'Has_Tax_Credit']

In [8]:
target_col = "Account_Number"
target_summary = pd.DataFrame({
    "part": ["Training", "Test"],
    "unique_classes": [train_df[target_col].nunique(), test_df[target_col].nunique()],
    "min_class_count": [
        train_df[target_col].value_counts().min(),
        test_df[target_col].value_counts().min()
    ]
})
display(target_summary)

,part,unique_classes,min_class_count
0,Training,27,1
1,Test,24,1


In [9]:
X_train = train_df[selected_features].copy()
X_test = test_df[selected_features].copy()
y_train = train_df[target_col].copy()
y_test = test_df[target_col].copy()

In [10]:
X_train["Combined_Text"] = (
    X_train["Item_Description"].fillna("") + " " +
    X_train["Activity_Description"].fillna("")
)

X_test["Combined_Text"] = (
    X_test["Item_Description"].fillna("") + " " +
    X_test["Activity_Description"].fillna("")
)

In [11]:
final_text_feature = "Combined_Text"
final_numeric_features = numeric_features.copy()
final_features = [final_text_feature] + final_numeric_features
final_features

['Combined_Text',
 'Item_Value',
 'Item_Quantity',
 'Taxes_Credit_%',
 'Taxes_Credit_Amount',
 'Month',
 'Item_Description_Length',
 'Item_Description_Word_Count',
 'Has_Tax_Credit']

In [12]:
X_train_final = X_train[[final_text_feature] + final_numeric_features].copy()
X_test_final = X_test[[final_text_feature] + final_numeric_features].copy()
display(X_train_final.head())

,Combined_Text,Item_Value,Item_Quantity,Taxes_Credit_%,Taxes_Credit_Amount,Month,Item_Description_Length,Item_Description_Word_Count,Has_Tax_Credit
0,review of federal tax obligations business man...,2000,1,0,0.0,8,33,5,0
1,assistance with flight and hotel arrangements ...,500,1,0,0.0,8,45,6,0
2,legal advisory in labor lawsuit legal services,2000,1,0,0.0,8,31,5,0
3,consulting on zero-based budgeting business ma...,15000,1,0,0.0,8,34,4,0
4,provision of 2 security guards for gate electr...,15000,1,0,0.0,8,39,7,0


In [13]:
text_preprocessor = Pipeline(steps=[
    ("extract_text", FunctionTransformer(squeeze_text_column, validate=False)),
    ("tfidf", TfidfVectorizer(
        lowercase=False,
        ngram_range=(1, 2),
        min_df=2
    ))
])

In [14]:
numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_preprocessor, [final_text_feature]),
        ("num", numeric_preprocessor, final_numeric_features)
    ]
)

In [16]:
X_train_processed = preprocessor.fit_transform(X_train_final)
X_test_processed = preprocessor.transform(X_test_final)

In [17]:
shape_summary = pd.DataFrame({
    "matrix": ["X_train_processed", "X_test_processed"],
    "rows": [X_train_processed.shape[0], X_test_processed.shape[0]],
    "cols": [X_train_processed.shape[1], X_test_processed.shape[1]]
})

display(shape_summary)

,matrix,rows,cols
0,X_train_processed,292,318
1,X_test_processed,51,318


In [18]:
train_classes = set(y_train.unique())
test_classes = set(y_test.unique())
class_diff_df = pd.DataFrame({
    "group": ["only_in_train", "only_in_test"],
    "classes": [sorted(train_classes - test_classes), sorted(test_classes - train_classes)]
})

display(class_diff_df)

,group,classes
0,only_in_train,"[1001002, 4000013, 5000017]"
1,only_in_test,[]


In [19]:
target_check = pd.DataFrame({
    "target_name": [target_col],
    "train_rows": [len(y_train)],
    "test_rows": [len(y_test)],
    "train_unique_classes": [y_train.nunique()],
    "test_unique_classes": [y_test.nunique()]
})

display(target_check)

,target_name,train_rows,test_rows,train_unique_classes,test_unique_classes
0,Account_Number,292,51,27,24


In [20]:
type(X_train_processed), type(X_test_processed)

(scipy.sparse._csr.csr_matrix, scipy.sparse._csr.csr_matrix)

In [21]:
sparse.save_npz("../data/modified/X_train_processed.npz", X_train_processed)
sparse.save_npz("../data/modified/X_test_processed.npz", X_test_processed)
y_train.to_csv("../data/modified/y_train.csv", index=False)
y_test.to_csv("../data/modified/y_test.csv", index=False)
joblib.dump(preprocessor, "../artifacts/preprocessor.joblib")

['../artifacts/preprocessor.joblib']

## Итоги preprocessing

На данном этапе была выполнена базовая подготовка данных для baseline-модели классификации:

1. Колонка `Date` была преобразована в формат datetime, после чего из нее были извлечены простые календарные признаки.
2. Для текстовых признаков была выполнена минимальная очистка: приведение к строковому типу, удаление лишних пробелов и перевод в нижний регистр.
3. Был сформирован объединенный текстовый признак `Combined_Text`, включающий `Item_Description` и `Activity_Description`.
4. Для baseline-модели были отобраны текстовые и числовые признаки без использования потенциально протекающих или идентификаторных полей.
5. Для текста был построен TF-IDF pipeline, а для числовых признаков — pipeline с заполнением пропусков и масштабированием.
6. Все преобразования были обучены только на training-части выборки и затем применены к test-части, что соответствует корректной схеме машинного обучения.